# Split preprocessed ADME data

---

This notebook performs the scaffold-based train/calibration/test split of the preprocessed ADME public dataset (produced by `Prepare_data.ipynb`).

**Note:** This notebook relies on `chemprop`, so it must be run using the `chemprop-env` conda environment.

In [1]:
import pandas as pd

from rdkit import Chem
from chemprop.data.splitting import make_split_indices

In [2]:
# Load preprocessed data
inputFile = "data/ADME_public_set_3521_preprocessed.csv"
df_preprocessed = pd.read_csv(inputFile)
df_preprocessed.head()

,Id,Structure,rLM LogCLint,hLM LogCLint,MDCK-MDR1_LogER,LogFu-Rat,LogFu-Human
0,Mol1,CNc1cc(Nc2cccn(-c3ccccn3)c2=O)nn2c(C(=O)N[C@@H...,1.392169,0.675687,1.493167,-1.481486,-1.008774
1,Mol2,CCOc1cc2nn(CCC(C)(C)O)cc2cc1NC(=O)c1cccc(C(F)F)n1,1.027920,0.675687,1.040780,-1.731656,-1.900319
2,Mol3,CN(c1ncc(F)cn1)[C@H]1CCCNC1,1.027920,0.675687,-0.358806,0.000000,0.000000
3,Mol4,CC(C)(Oc1ccc(-c2cnc(N)c(-c3ccc(Cl)cc3)c2)cc1)C...,1.027920,0.675687,1.026662,-3.403403,-3.158015
4,Mol5,CC(C)(O)CCn1cc2cc(NC(=O)c3cccc(C(F)(F)F)n3)c(C...,1.629093,0.996380,1.010597,-0.907736,-0.984389


In [3]:
# Generate RDKit molecule objects
df_preprocessed['mol'] = df_preprocessed['Structure'].apply(lambda x: Chem.MolFromSmiles(x))

In [4]:
# Define the endpoints dictionary
endpoints_dict = {'CLint': ['rLM LogCLint', 'hLM LogCLint'],
                  'MDR1': ['MDCK-MDR1_LogER'],
                  'PPB': ['LogFu-Rat', 'LogFu-Human']}

for endpoint, models_list in endpoints_dict.items():
    print(f'\n### {endpoint} ###\n')

    # Select endpoint data
    df_endpoint = df_preprocessed[['Id', 'Structure', 'mol'] + models_list]
    df_endpoint = df_endpoint.dropna(subset=models_list, how='all').reset_index(drop=True)
    print(df_endpoint.shape)

    # Split data into training, calibration and test sets (scaffold-based)
    train_idxs, cal_idxs, val_idxs = make_split_indices(df_endpoint['mol'], split='SCAFFOLD_BALANCED', sizes=(0.5, 0.3, 0.2))
    train_idxs, cal_idxs, val_idxs = list(train_idxs[0]), list(cal_idxs[0]), list(val_idxs[0])
    df_endpoint['Subset'] = ''
    df_endpoint.loc[train_idxs, 'Subset'] = 'Training'
    df_endpoint.loc[cal_idxs, 'Subset'] = 'Calibration'
    df_endpoint.loc[val_idxs, 'Subset'] = 'Validation'

    # Select training data and further split it into training and validation
    # sets for model training and early stopping (scaffold-based)
    training_data = df_endpoint.loc[df_endpoint['Subset'] == 'Training'].reset_index(drop=True)
    test_data = df_endpoint.loc[df_endpoint['Subset'] != 'Training'].reset_index(drop=True)
    train_idxs, _, val_idxs = make_split_indices(training_data['mol'], split='SCAFFOLD_BALANCED', sizes=(0.9, 0, 0.1))
    train_idxs, val_idxs = list(train_idxs[0]), list(val_idxs[0])
    training_data['split'] = ''
    training_data.loc[train_idxs, 'split'] = 'train'
    training_data.loc[val_idxs, 'split'] = 'val'
    test_data['split'] = 'test'
    df_endpoint = pd.concat([training_data, test_data], ignore_index=True)
    summary_time_split = pd.DataFrame([df_endpoint[f'Subset'].value_counts()[['Training','Calibration','Validation']], (df_endpoint[f'Subset'].value_counts()/df_endpoint.shape[0]*100)[['Training','Calibration','Validation']]], index=['# Cpds','% Cpds'])
    print(summary_time_split.round())
    summary_time_split = pd.DataFrame([df_endpoint[f'split'].value_counts()[['train','val','test']], (df_endpoint[f'split'].value_counts()/df_endpoint.shape[0]*100)[['train','val','test']]], index=['# Cpds','% Cpds'])
    print(summary_time_split.round())

    # Save preprocessed endpoint dataset with data subsets and splits
    df_endpoint.to_csv(f'data/{endpoint}_dataset_with_splits.csv')

The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)



### CLint ###

(3089, 5)


/chemotargets/shared/conda/envs/chemprop-env/lib/python3.11/site-packages/astartes/samplers/extrapolation/scaffold.py:48: NoMatchingScaffold: No matching scaffold was found for the 2 molecules corresponding to indices {2304, 1823}
  warnings.warn(
The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)


Subset  Training  Calibration  Validation
# Cpds    1546.0        926.0       617.0
% Cpds      50.0         30.0        20.0
split    train    val    test
# Cpds  1392.0  154.0  1543.0
% Cpds    45.0    5.0    50.0

### MDR1 ###

(2639, 4)


/chemotargets/shared/conda/envs/chemprop-env/lib/python3.11/site-packages/astartes/samplers/extrapolation/scaffold.py:48: NoMatchingScaffold: No matching scaffold was found for the 1 molecules corresponding to indices {2048}
  warnings.warn(
The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
/chemotargets/shared/conda/envs/chemprop-env/lib/python3.11/site-packages/astartes/samplers/extrapolation/scaffold.py:48: NoMatchingScaffold: No matching scaffold was found for the 1 molecules corresponding to indices {1017}
  warnings.warn(
The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)


Subset  Training  Calibration  Validation
# Cpds    1321.0        791.0       527.0
% Cpds      50.0         30.0        20.0
split    train    val    test
# Cpds  1189.0  132.0  1318.0
% Cpds    45.0    5.0    50.0

### PPB ###

(206, 5)
Subset  Training  Calibration  Validation
# Cpds     104.0         61.0        41.0
% Cpds      50.0         30.0        20.0
split   train   val   test
# Cpds   94.0  10.0  102.0
% Cpds   46.0   5.0   50.0
